In [1]:
import sys
import json
import pandas as pd

sys.path.append("..")

from etl_pipeline_local import get_goal_home_ht_table

In [2]:
# Data upload - do not implement
# Bet setup
target_col = "goal_home_ht"

# Load data
df_loaded = get_goal_home_ht_table()
df_loaded = df_loaded.sort_values('time').reset_index(drop=True)
df = df_loaded.copy()

In [3]:
def get_mask(df, params_dict):
    mask = pd.Series(True, index=df.index)
    features = set()

    for key in params_dict:
        for suffix in ["_cat", "_use_min", "_use_max", "_min", "_max", "_include_missing"]: # ,   "_min_idx", "_max_idx", 
             if key.endswith(suffix):
                features.add(key[:-len(suffix)])
                break

    for feat in features:
        s = df[feat]

        include_missing = params_dict.get(f"{feat}_include_missing", False)
        use_min = params_dict.get(f"{feat}_use_min", True)
        use_max = params_dict.get(f"{feat}_use_max", True)

        feat_mask = pd.Series(True, index=df.index)

        # Categorical filtering
        if f"{feat}_cat" in params_dict:
            feat_mask &= s.isin(params_dict[f"{feat}_cat"])

        # Numerical filtering
        if f"{feat}_min" in params_dict and use_min:
            feat_mask &= s >= params_dict[f"{feat}_min"]

        if f"{feat}_max" in params_dict and use_max:
            feat_mask &= s <= params_dict[f"{feat}_max"]

        if include_missing:
            feat_mask = feat_mask | s.isna()
        else:
            feat_mask = feat_mask & s.notna()

        if params_dict[f"use_{feat}"]:
            mask &= feat_mask

    return mask


def round_to_step(x, step):
    """
    Arrotonda x a un multiplo di 'step'.
    """
    try:
        if not step:
            return str(x)
        elif step <= 0:
            raise ValueError("step deve essere > 0")
        else:
            ratio = x / step
            return round(ratio) * step
    
    except Exception as e: 
        return None

In [4]:
with open("../../strategies/strategies.json", "r", encoding="utf-8") as f:
    data = json.load(f)

params_dict = data[target_col]['params_dict']
feature_bins_map = data[target_col]['feature_bins_map']

# Feature Engineering
df = df[df["underOver_quote_currentO"] >= 1.4]

df_binned = df.copy()

for feat, step in feature_bins_map.items():
    df[feat] = [round_to_step(x, step) for x in df[feat]]

    if isinstance(step, int):
        df[feat] = df[feat].astype("Int64")


mask = get_mask(df, params_dict)

df_filtered = df[mask]
df_filtered.head()


,time,chance1x2_chance_p1,chance1x2_chance_px,chance1x2_chance_p2,chance1x2_chance_p1x,chance1x2_chance_p2x,chance1x2_chance_p12,chance1x2_chance_pHt1,chance1x2_chance_pHtx,chance1x2_chance_pHt2,...,underOver_flashback_under35,underOver_flashback_over35,team_goal,team_goalHt,team_corner,chance1x2_xg_home,chance1x2_xg_away,chance1x2_xg_total,chance1x2_xg,goal_home_ht
1762,2025-08-30 12:00:00,57.0,22.4,20.6,79.4,43.0,77.6,35.4,45.1,19.5,...,69.2,30.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
1770,2025-08-30 13:30:00,67.1,19.6,13.3,86.7,32.9,80.4,48.5,34.2,17.3,...,72.0,28.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
1772,2025-08-30 13:35:00,47.9,26.4,25.7,74.3,52.1,73.6,34.3,38.8,26.9,...,58.7,41.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
1808,2025-08-30 17:00:00,76.0,16.4,7.6,92.4,24.0,83.6,66.4,25.0,8.6,...,60.3,39.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
1871,2025-08-31 14:00:00,68.9,19.3,11.8,88.2,31.1,80.7,49.5,40.6,9.9,...,64.6,35.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True


In [5]:
df_filtered.shape

(534, 139)